<a href="https://colab.research.google.com/github/AdamHany03/UniversityWork/blob/main/CNN%2BLSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
import mplfinance as mpf
import yfinance as yf
import pandas as pd
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, Sequential, optimizers, callbacks, utils

seq_len = 5
h, w, c = 64, 64, 3
classes = 3
win_size = 5
fut_p = 1

tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA"]
start, end = "2014-01-01", "2024-01-01"

In [11]:
all_data = []
for t in tickers:
    df = yf.download(t, start=start, end=end, auto_adjust=True, progress=False)
    if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)
    df = df[["Open", "High", "Low", "Close", "Volume"]].astype(float)
    df['Ticker'] = t
    all_data.append(df)

df_combined = pd.concat(all_data)

In [12]:
os.makedirs("charts", exist_ok=True)
X_imgs, y_labs = [], []

for t in tickers:
    t_df = df_combined[df_combined['Ticker'] == t].copy()
    limit = min(len(t_df) - win_size - fut_p + 1, 600)
    for i in range(limit):
        chunk = t_df.iloc[i : i + win_size]
        fname = f"charts/{t}_{i}.png"
        fig, ax = plt.subplots(figsize=(w/100, h/100), dpi=100)
        fig.subplots_adjust(0, 0, 1, 1)
        mpf.plot(chunk, type='candle', style='yahoo', ax=ax)
        ax.set_axis_off()
        plt.savefig(fname, bbox_inches='tight', pad_inches=0)
        plt.close(fig)

        img = cv2.imread(fname)
        img = cv2.resize(img, (w, h))
        X_imgs.append(img.astype('float32') / 255.0)

        c_val = t_df['Close'].iloc[i + win_size - 1]
        f_val = t_df['Close'].iloc[i + win_size + fut_p - 1]
        y_labs.append(0 if f_val > c_val else (1 if f_val < c_val else 2))

X, y = [], []
for i in range(len(X_imgs) - seq_len + 1):
    X.append(X_imgs[i : i + seq_len])
    y.append(y_labs[i + seq_len - 1])

X = np.array(X)
y = utils.to_categorical(np.array(y), classes)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)

In [13]:
model = Sequential([
    layers.Input(shape=(seq_len, h, w, c)),
    layers.TimeDistributed(layers.Conv2D(64, (3,3), padding='same', activation='relu')),
    layers.TimeDistributed(layers.BatchNormalization()),
    layers.TimeDistributed(layers.MaxPooling2D((2,2))),
    layers.TimeDistributed(layers.Conv2D(128, (3,3), padding='same', activation='relu')),
    layers.TimeDistributed(layers.BatchNormalization()),
    layers.TimeDistributed(layers.MaxPooling2D((2,2))),
    layers.TimeDistributed(layers.Conv2D(256, (3,3), padding='same', activation='relu')),
    layers.TimeDistributed(layers.Flatten()),
    layers.LSTM(256, return_sequences=False),
    layers.Dropout(0.5),
    layers.Dense(128, activation='relu'),
    layers.Dense(classes, activation='softmax')
])

model.compile(optimizer=optimizers.Adam(0.0001), loss='categorical_crossentropy', metrics=['accuracy'])
es = callbacks.EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True)

model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.15, callbacks=[es])

loss, acc = model.evaluate(X_test, y_test)
print(f"Final Accuracy: {acc:.4f}")

Epoch 1/50
68/68 ━━━━━━━━━━━━━━━━━━━━ 21s 226ms/step - accuracy: 0.4935 - loss: 0.7957 - val_accuracy: 0.4555 - val_loss: 0.7177
Epoch 2/50
68/68 ━━━━━━━━━━━━━━━━━━━━ 14s 199ms/step - accuracy: 0.4889 - loss: 0.7515 - val_accuracy: 0.4555 - val_loss: 0.7384
Epoch 3/50
68/68 ━━━━━━━━━━━━━━━━━━━━ 14s 199ms/step - accuracy: 0.4935 - loss: 0.7486 - val_accuracy: 0.4555 - val_loss: 0.7082
Epoch 4/50
68/68 ━━━━━━━━━━━━━━━━━━━━ 14s 199ms/step - accuracy: 0.4898 - loss: 0.7405 - val_accuracy: 0.4555 - val_loss: 0.7205
Epoch 5/50
68/68 ━━━━━━━━━━━━━━━━━━━━ 14s 200ms/step - accuracy: 0.5009 - loss: 0.7349 - val_accuracy: 0.4555 - val_loss: 0.7141
Epoch 6/50
68/68 ━━━━━━━━━━━━━━━━━━━━ 14s 201ms/step - accuracy: 0.5037 - loss: 0.7305 - val_accuracy: 0.4555 - val_loss: 0.7240
Epoch 7/50
68/68 ━━━━━━━━━━━━━━━━━━━━ 14s 207ms/step - accuracy: 0.4958 - loss: 0.7311 - val_accuracy: 0.4581 - val_loss: 0.7145
Epoch 8/50
68/68 ━━━━━━━━━━━━━━━━━━━━ 14s 201ms/step - accuracy: 0.5162 - loss: 0.7219 - val_accu